In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

In [3]:
df = pd.read_csv("Language_translation_models_dataset - Language_translation_models_dataset.csv")
df['cleaned_target'] = df['Target_Text'].fillna('').str.replace(r'^\[.*?\s+translation\]\s*', '', regex = True)

In [ ]:
placeholders = r"^i don't know$|^idk$|^unknown$|^n/a$|^none$|^placeholder$|^missing$|^blank$|^not collected$"

summary = []

for col in df.columns:
    str_col = df[col].astype(str).str.strip()

    null = df[col].isnull().sum()
    blank = (str_col == '').sum() - null
    placeholder_count = str_col.str.contains(placeholders, case=False, na=False).sum()
    total = null + max(0, blank) + placeholder_count
    percentage = (total / len(df)) * 100

    summary.append({
        'Column': col,
        'Nulls': null,
        'Blank Strings': max(0, blank),
        'Placeholder Text': placeholder_count,
        'Total': total,
        'Percentage (%)': round(percentage, 2)
    })

missing_df = pd.DataFrame(summary)

print('MISSING/UNKNOWN VALUES')
print(missing_df.to_string(index=False))

In [ ]:
source_languages = df['Source_Language'].dropna().unique().tolist()
target_languages = df['Target_Language'].dropna().unique().tolist()
all_languages = list(set(source_languages + target_languages))

print("NUMBER OF LANGUAGES")
print(f'Source Language(s): {source_languages}')
print(f'Target Languages ({len(target_languages)}): {target_languages}')
print(f'Total Languanges: {len(all_languages)}')

In [ ]:
df['src_word_count'] = df['Source_Text'].fillna('').apply(lambda x: len(str(x).split()))
df['tgt_word_count'] = df['cleaned_target'].apply(lambda x: len(str(x).split()))

print('LENGTH SUMMARY STATISTICS')
print(df[['src_word_count', 'tgt_word_count']].describe())

In [ ]:
df_melted = df.melt(
    id_vars=['Target_Language'],
    value_vars=['src_word_count', 'tgt_word_count'],
    var_name='Text_Type',
    value_name='Word_Count'
)

df_melted['Text_Type'] = df_melted['Text_Type'].replace({
    'src_word_count': 'Source',
    'tgt_word_count': 'Target Language'
})

plt.figure(figsize=(12, 6))
sns.boxplot(
    data = df_melted,
    x = 'Target_Language',
    y = 'Word_Count',
    hue = 'Text_Type',
    palette = 'Set2'
)

plt.title('Source and Target Word Comparison per Target Language', fontsize = 14, fontweight = 'bold')
plt.xlabel('Target Language', fontsize = 12)
plt.ylabel('Number of Words', fontsize = 12)
plt.legend(title = 'Text Version')
plt.tight_layout()
plt.show()

In [ ]:
lang_count = df['Target_Language'].value_counts()

print('SENTENCES PER LANGUAGE')
print(lang_count)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=lang_count.values, y = lang_count.index, palette= 'viridis')
plt.title('Sentences per Target Language')
plt.xlabel('Sentence Count')
plt.ylabel('Target Language')

for i, count in enumerate(lang_count.values):
    ax.text(count + 2, i, str(count), va = 'center', fontweight = 'bold')

plt.tight_layout()
plt.show()

In [ ]:
common_phrases = df['Source_Text'].value_counts()

print('TOP COMMON PHRASES')
print(common_phrases.head(10))

shared_matrix = pd.crosstab(df['Source_Text'], df['Target_Language'])

plt.figure(figsize=(10, 6))
sns.heatmap(shared_matrix, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Shared Phrases across Target Languages')
plt.xlabel('Target Language')
plt.ylabel('Source Phrase')
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(index='Source_Text', columns='Target_Language', values='cleaned_target', aggfunc='first')
languages = df['Target_Language'].dropna().unique()
overlap = pd.DataFrame(0, index=languages, columns=languages)

for l1 in languages:
    for l2 in languages:
        if l1 == l2:
            overlap.loc[l1, l2] = pivot[l1].notnull().sum()
        else:
            matches = (pivot[l1].notnull()) & (pivot[l2].notnull()) & (pivot[l1] == pivot[l2])
            overlap.loc[l1, l2] = matches.sum()

print('INDENTICAL TRANSLATIONS')
print(overlap)

plt.figure(figsize=(9,7))
sns.heatmap(overlap, annot=True, fmt='d', cmap='YlGnBu', cbar=True, linewidths=0.5)

plt.title('Number of Identical Translations Shared Between Target Languages', fontweight='bold')
plt.xlabel('Target Language', fontsize=11)
plt.ylabel('Target Language', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
formality = df['Formality'].value_counts()

formality_by_lang = pd.crosstab(df['Target_Language'], df['Formality'])

print('FORMALITY ANALYSIS')
print(formality_by_lang)

formality_by_lang.plot(kind='bar', figsize=(9, 5), color=['#2b5c8f', "#f86a05"])
plt.title('Formality Levels per target Language')
plt.xlabel('Target Language')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Formality')
plt.tight_layout()
plt.show()

In [ ]:
region = pd.crosstab(df['Target_Language'], df['Speaker_Region'])
print(region)

region.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set2')
plt.title('Speaker Region Distribution per Target Language')
plt.xlabel('Target Language')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
domain_by_lang = pd.crosstab(df['Domain'], df['Target_Language'])

print('DOMAIN DISTRIBUTION')
print(domain_by_lang)

plt.figure(figsize=(10, 6))
sns.heatmap(domain_by_lang, annot=True, fmt='d', cmap='Greens', cbar=False)
plt.title('Domain Distribution across Target Languages')
plt.xlabel('Target Language')
plt.ylabel('Domain')
plt.tight_layout()
plt.show()